In [1]:
import pandas as pd
from pathlib import Path

# Path to your GTFS data folder
GTFS_PATH = Path(r"F:\P Projects 2026\Berlin Project\data")

# Read main GTFS files
stops = pd.read_csv(GTFS_PATH / "stops.txt")
stop_times = pd.read_csv(GTFS_PATH / "stop_times.txt", dtype={"arrival_time": str, "departure_time": str}, low_memory=False)
trips = pd.read_csv(GTFS_PATH / "trips.txt")
routes = pd.read_csv(GTFS_PATH / "routes.txt")
calendar = pd.read_csv(GTFS_PATH / "calendar.txt")
calendar_dates = pd.read_csv(GTFS_PATH / "calendar_dates.txt")


In [2]:

monday_trips = calendar[calendar['monday'] == 1]['service_id']

# Include added services from calendar_dates
added_services = calendar_dates[calendar_dates['exception_type'] == 1]['service_id']

# Combine all active service_ids
active_service_ids = pd.concat([monday_trips, added_services]).unique()

# Filter trips by these services
active_trips = trips[trips['service_id'].isin(active_service_ids)]

# Filter stop_times for active trips
active_stop_times = stop_times[stop_times['trip_id'].isin(active_trips['trip_id'])]


In [3]:
# Merge stop_times with stops to get station names and coordinates
stop_data = active_stop_times.merge(
    stops[['stop_id','stop_name','stop_lat','stop_lon']],
    on='stop_id', how='left'
)

# Count number of arrivals per station
busiest_stations = stop_data.groupby(['stop_name','stop_lat','stop_lon'], as_index=False)['trip_id'].count()
busiest_stations.rename(columns={'trip_id':'num_arrivals'}, inplace=True)

# Sort descending
busiest_stations = busiest_stations.sort_values('num_arrivals', ascending=False)


In [4]:
busiest_stations.head(10)


,stop_name,stop_lat,stop_lon,num_arrivals
9929,Hertzallee (Berlin),52.509228,13.332082,6509
19963,S+U Zoologischer Garten Bhf (Berlin),52.506741,13.333567,6102
9928,Hertzallee (Berlin),52.508833,13.333586,5255
23585,U Kurt-Schumacher-Platz (Berlin),52.563324,13.327342,3715
23586,U Kurt-Schumacher-Platz (Berlin),52.563346,13.326896,3539
1923,"Berlin, Moritzstr.",52.539015,13.200849,3503
19389,S Potsdamer Platz Bhf/Voßstr. (Berlin),52.510168,13.376763,3425
1858,"Berlin, Invalidenpark",52.528659,13.376728,3417
19937,S+U Wittenau (Berlin),52.595542,13.335021,3387
1921,"Berlin, Moritzstr.",52.538297,13.200741,3386


In [3]:
import pandas as pd
import geopandas as gpd
import networkx as nx
from shapely.geometry import Point
import matplotlib.pyplot as plt


In [4]:
gtfs_path = r"F:/P Projects 2026/Berlin Project/data/"


In [5]:
stops = pd.read_csv(gtfs_path + "stops.txt")
stop_times = pd.read_csv(gtfs_path + "stop_times.txt")
trips = pd.read_csv(gtfs_path + "trips.txt")
routes = pd.read_csv(gtfs_path + "routes.txt")
calendar = pd.read_csv(gtfs_path + "calendar.txt")


C:\Users\dell\AppData\Local\Temp\ipykernel_8284\64123888.py:2: DtypeWarning: Columns (0: stop_headsign) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv(gtfs_path + "stop_times.txt")


In [8]:
stops.head()


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code,zone_id,level_id
0,de:11000:900007104::2,5,S Nordbahnhof (Berlin),Tram u. Bussteig Invalidenstraße ht. Gartenstraße,52.531683,13.388813,0,NaN,0,NaN,5555 S Nordbahnhof (Berlin),4.0
1,de:11000:900100007::3,NaN,S Oranienburger Str. (Berlin),Ersatzhalt Tucholskystraße vor Oranienburger S...,52.524724,13.392833,0,NaN,0,NaN,5555 S Oranienburger Str. (Berlin),4.0
2,de:12070:900215110:1:50,NaN,"Bad Wilsnack, Bahnhof",Bahnsteig Gleis 2,52.960114,11.949402,0,de:12070:900215110,1,2,"4533 Bad Wilsnack, Bahnhof",50.0
3,de:12070:900215110:2:51,NaN,"Bad Wilsnack, Bahnhof",Bahnsteig Gleis 3,52.960219,11.949528,0,de:12070:900215110,1,3,"4533 Bad Wilsnack, Bahnhof",50.0
4,de:12062:900415465:1:50,NaN,"Prösen, Bahnhof",Bahnsteig Gleis 1,51.434919,13.488216,0,de:12062:900415465,1,1,"7958 Prösen, Bahnhof",4.0


In [9]:
stops[stops["stop_name"].str.contains("Berlin", case=False, na=False)].head(20)


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code,zone_id,level_id
0,de:11000:900007104::2,5,S Nordbahnhof (Berlin),Tram u. Bussteig Invalidenstraße ht. Gartenstraße,52.531683,13.388813,0,NaN,0,NaN,5555 S Nordbahnhof (Berlin),4.0
1,de:11000:900100007::3,NaN,S Oranienburger Str. (Berlin),Ersatzhalt Tucholskystraße vor Oranienburger S...,52.524724,13.392833,0,NaN,0,NaN,5555 S Oranienburger Str. (Berlin),4.0
28,de:11000:900160004:3:54,NaN,S+U Lichtenberg Bhf (Berlin),Bahnsteig Gleis 17,52.510002,13.496490,0,de:11000:900160004,0,17,5656 S+U Lichtenberg Bhf (Berlin),102.0
29,de:11000:900160004:2:52,NaN,S+U Lichtenberg Bhf (Berlin),Bahnsteig Gleis 15,52.510275,13.497433,0,de:11000:900160004,0,15,5656 S+U Lichtenberg Bhf (Berlin),102.0
30,de:11000:900160004:2:53,NaN,S+U Lichtenberg Bhf (Berlin),Bahnsteig Gleis 16,52.510367,13.497302,0,de:11000:900160004,0,16,5656 S+U Lichtenberg Bhf (Berlin),102.0
31,de:11000:900160004:3:55,NaN,S+U Lichtenberg Bhf (Berlin),Bahnsteig Gleis 20,52.510088,13.496357,0,de:11000:900160004,0,20,5656 S+U Lichtenberg Bhf (Berlin),102.0
32,de:11000:900160004:4:56,NaN,S+U Lichtenberg Bhf (Berlin),Bahnsteig Gleis 21,52.510126,13.496304,0,de:11000:900160004,0,21,5656 S+U Lichtenberg Bhf (Berlin),102.0
33,de:11000:900160004:4:57,NaN,S+U Lichtenberg Bhf (Berlin),Bahnsteig Gleis 22,52.510215,13.496164,0,de:11000:900160004,0,22,5656 S+U Lichtenberg Bhf (Berlin),102.0
68,de:11000:900120005:1:50,NaN,S Ostbahnhof (Berlin),Bahnsteig Gleis 1,52.510244,13.434586,0,de:11000:900120005,0,1,5555 S Ostbahnhof (Berlin),107.0
69,de:11000:900120005:2:51,NaN,S Ostbahnhof (Berlin),Bahnsteig Gleis 2,52.510179,13.434942,0,de:11000:900120005,0,2,5555 S Ostbahnhof (Berlin),107.0


In [6]:
origin_stop_id = "000300258512"
origin_stop_id in stops["stop_id"].values



True

In [8]:
def time_to_seconds(t):
    if pd.isna(t):
        return None
    try:
        h, m, s = str(t).split(":")
        return int(h)*3600 + int(m)*60 + int(s)
    except:
        return None

stop_times["arrival_sec"] = stop_times["arrival_time"].apply(time_to_seconds)
stop_times = stop_times.dropna(subset=["arrival_sec"])  # Remove rows with missing times

In [7]:
stop_times = stop_times.sort_values(
    ["trip_id", "stop_sequence"]
)


In [9]:
edges = []
count = 0

for trip_id, group in stop_times.groupby("trip_id"):
    group = group.sort_values("stop_sequence")
    
    for i in range(len(group) - 1):
        try:
            from_stop = group.iloc[i]["stop_id"]
            to_stop = group.iloc[i+1]["stop_id"]
            from_time = group.iloc[i]["arrival_sec"]
            to_time = group.iloc[i+1]["arrival_sec"]
            
            if pd.notna(from_time) and pd.notna(to_time):
                travel_time = int(to_time - from_time)
                
                if travel_time > 0 and travel_time < 3600:  # Between 0 and 60 minutes
                    edges.append((from_stop, to_stop, travel_time))
                    count += 1
        except Exception as e:
            continue

print(f"Created {count} edges from {len(stop_times)} stop times")

Created 5376496 edges from 5633240 stop times


In [10]:
import networkx as nx

G = nx.DiGraph()

for u, v, w in edges:
    if pd.notna(u) and pd.notna(v):  # Ensure valid nodes
        G.add_edge(u, v, weight=w)

print(f"Graph created with {len(G.nodes)} nodes and {len(G.edges)} edges")
print(f"Origin stop {origin_stop_id} in graph: {origin_stop_id in G.nodes}")

Graph created with 26413 nodes and 40829 edges
Origin stop 000300258512 in graph: False


In [11]:
max_time = 60 * 60

if origin_stop_id not in G.nodes:
    print(f"Error: Origin stop {origin_stop_id} not in graph!")
    print(f"Finding closest stop in graph...")
    # Try to find the stop in the stops dataframe
    origin_info = stops[stops["stop_id"] == origin_stop_id]
    if len(origin_info) > 0:
        print(f"Stop found: {origin_info['stop_name'].values[0]}")
        print(f"Available nodes in graph: {len(G.nodes)}")
        # Get first available node from stops in Berlin
        berlin_stops = stops[stops['stop_name'].str.contains('Berlin', case=False, na=False)]
        available_in_graph = berlin_stops[berlin_stops['stop_id'].isin(G.nodes)]
        if len(available_in_graph) > 0:
            origin_stop_id = available_in_graph.iloc[0]['stop_id']
            print(f"Using alternative origin: {available_in_graph.iloc[0]['stop_name']} ({origin_stop_id})")
else:
    print(f"Origin stop {origin_stop_id} found in graph")

lengths = nx.single_source_dijkstra_path_length(
    G,
    origin_stop_id,
    cutoff=max_time,
    weight="weight"
)

print(f"Found {len(lengths)} reachable stops within 60 minutes")

Error: Origin stop 000300258512 not in graph!
Finding closest stop in graph...
Stop found: S+U Berlin Hauptbahnhof
Available nodes in graph: 26413
Using alternative origin: S Nordbahnhof (Berlin) (de:11000:900007104::2)
Found 5336 reachable stops within 60 minutes


In [13]:
len(G.nodes), len(G.edges)


(26413, 40829)

In [15]:
import geopandas as gpd
from shapely.geometry import Point

reachable_stops = stops[stops["stop_id"].isin(lengths.keys())].copy()
reachable_stops["travel_time_min"] = reachable_stops["stop_id"].map(lengths) / 60

gdf_stops = gpd.GeoDataFrame(
    reachable_stops,
    geometry=gpd.points_from_xy(
        reachable_stops.stop_lon,
        reachable_stops.stop_lat
    ),
    crs="EPSG:4326"
)

gdf_stops.head()


,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code,zone_id,level_id,travel_time_min,geometry
0,de:11000:900007104::2,5,S Nordbahnhof (Berlin),Tram u. Bussteig Invalidenstraße ht. Gartenstraße,52.531683,13.388813,0,NaN,0,NaN,5555 S Nordbahnhof (Berlin),4.0,0.0,POINT (13.38881 52.53168)
2154,de:11000:900073253::2,NaN,Ahrensdorfer Str. (Berlin),Bushalt Marienfelder Allee vor Ahrensdorfer St...,52.409158,13.359291,0,NaN,0,NaN,5656 Ahrensdorfer Str. (Berlin),4.0,56.5,POINT (13.35929 52.40916)
2155,de:11000:900073254::1,NaN,Baußnernweg (Berlin),Bushalt Marienfelder Allee vor Wippraer Weg,52.405521,13.356591,0,NaN,0,NaN,5656 Baußnernweg (Berlin),4.0,57.5,POINT (13.35659 52.40552)
2158,de:11000:900073203::1,NaN,Klausenburger Pfad (Berlin),Bushalt Marienfelder Allee vor Klausenburger Pfad,52.402828,13.353398,0,NaN,0,NaN,5656 Klausenburger Pfad (Berlin),4.0,58.5,POINT (13.3534 52.40283)
2159,de:11000:900072102::1,NaN,Gutspark Marienfelde (Berlin),Bushalt Nahmitzer Damm ht. Motzener Straße,52.409227,13.376924,0,NaN,0,NaN,5656 Gutspark Marienfelde (Berlin),4.0,53.5,POINT (13.37692 52.40923)
